# Algorithms: Counting Steps, Measuring Growth

Two algorithms can both be "correct" and yet one finishes in a blink while the other outlives the universe. This lab is about *why*: we will count steps, run timing experiments, draw growth curves, and implement the classics — recursion, sorting, and binary search — with instruments attached so you can *see* the costs.

**How to use this notebook:** run cells top to bottom (`Shift+Enter`); the plotting cells reuse measurements from earlier cells. Timings will differ from machine to machine (and browser to browser) — the *ratios* are what matter, and those are remarkably stable.

Prerequisite: functions, loops, and lists; the data-structures lab helps but isn't required.

## Counting steps

Before any theory, let's literally count. Here is a function that sums a list, instrumented with a step counter — one tick per loop iteration:

In [ ]:
def sum_with_counter(items):
    steps = 0
    total = 0
    for x in items:
        total += x
        steps += 1
    return total, steps

for n in [10, 100, 1000, 10000]:
    total, steps = sum_with_counter(list(range(n)))
    print(f"n = {n:>6}  ->  {steps:>6} steps")

No surprise: summing $n$ items takes $n$ steps. Ten times the data, ten times the work — **linear** growth.

Now a different job: find whether any two numbers in a list are equal, the obvious way — compare every pair:

In [ ]:
def count_pair_checks(items):
    steps = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            steps += 1                      # one comparison of a pair
    return steps

for n in [10, 100, 1000]:
    print(f"n = {n:>5}  ->  {count_pair_checks(list(range(n))):>8} pair checks")

Ten times the data now costs about a **hundred** times the work: $n$ items have $n(n-1)/2 \approx n^2/2$ pairs. That's **quadratic** growth, and it's the difference between "instant" and "coffee break" long before data gets big.

**Big-O notation** is the shorthand for these growth shapes. We say summing is $O(n)$ and all-pairs checking is $O(n^2)$: the notation keeps only how the step count *scales* with $n$, throwing away constant factors and smaller terms (so $n^2/2 + 3n$ is just $O(n^2)$). The usual suspects, best to worst:

| Big-O | Name | Feel |
|---|---|---|
| $O(1)$ | constant | same cost at any size |
| $O(\log n)$ | logarithmic | doubling $n$ adds one step |
| $O(n)$ | linear | doubling $n$ doubles the work |
| $O(n \log n)$ | linearithmic | good sorting lives here |
| $O(n^2)$ | quadratic | doubling $n$ quadruples the work |
| $O(2^n)$ | exponential | adding ONE item doubles the work |

## The doubling experiment

Here's the lab technique that turns Big-O from theory into measurement: **double the input size and watch what the running time does.** Linear code should take ~2x longer; quadratic code ~4x. Let's time real work with `time.perf_counter()` (a high-resolution stopwatch):

In [ ]:
import time

def time_it(func, arg):
    start = time.perf_counter()
    func(arg)
    return time.perf_counter() - start

lin_sizes, lin_times = [], []
for n in [25_000, 50_000, 100_000, 200_000, 400_000]:
    t = time_it(sum, list(range(n)))            # summing: O(n)
    lin_sizes.append(n)
    lin_times.append(t)
    ratio = "" if len(lin_times) < 2 else f"  ratio vs previous: {t / max(lin_times[-2], 1e-9):.1f}x"
    print(f"n = {n:>7}  {t * 1000:8.3f} ms{ratio}")

The ratios should hover around 2 (small sizes are noisy — the stopwatch and the computer's background chatter intrude; trust the larger sizes most).

Same experiment on the quadratic pair-checker — note how much *smaller* the inputs must be to keep it tolerable:

In [ ]:
quad_sizes, quad_times = [], []
for n in [50, 100, 200, 400]:
    t = time_it(count_pair_checks, list(range(n)))   # O(n^2)
    quad_sizes.append(n)
    quad_times.append(t)
    ratio = "" if len(quad_times) < 2 else f"  ratio vs previous: {t / max(quad_times[-2], 1e-9):.1f}x"
    print(f"n = {n:>4}  {t * 1000:8.3f} ms{ratio}")

Ratios near 4 — the quadratic fingerprint. The doubling experiment is genuinely how practitioners sanity-check performance: no theory needed, just two runs and a division.

## Seeing growth: the log-log plot

Plotting time against size on ordinary axes squashes everything interesting into the corner. The professional trick is a **log-log plot** (both axes logarithmic): every power law $t \approx c \cdot n^k$ becomes a *straight line whose slope is the exponent* $k$. Linear algorithms plot with slope 1, quadratic with slope 2.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(lin_sizes, [t * 1000 for t in lin_times], "o-", label="sum - $O(n)$, slope 1")
ax.loglog(quad_sizes, [t * 1000 for t in quad_times], "s-", label="pair checks - $O(n^2)$, slope 2")

ax.set_xlabel("input size n (log scale)")
ax.set_ylabel("time in ms (log scale)")
ax.set_title("The doubling experiment on log-log axes")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

Two roughly straight lines, the quadratic one visibly steeper — you are literally looking at the exponents. (Wobbles at the small end are timer noise; growth laws show their true face at scale.)

## Recursion: functions that call themselves

A **recursive** function solves a problem by calling itself on a smaller version, until a **base case** stops the descent. The textbook example: $n! = n \times (n-1)!$, with $1! = 1$. Let's add a `depth` parameter and indent the printout so the nesting becomes visible:

In [ ]:
def factorial(n, depth=0):
    indent = "    " * depth
    print(f"{indent}factorial({n})?")
    if n <= 1:
        print(f"{indent}base case -> 1")
        return 1
    result = n * factorial(n - 1, depth + 1)
    print(f"{indent}factorial({n}) = {result}")
    return result

print("answer:", factorial(5))

Read the indentation like a staircase: five calls descend to the base case, then the results climb back up, each level multiplying on its way out. Every indent level is a **stack frame** — memory the machine holds for a call that is still waiting. (The systems lab pushes on what happens when the staircase gets *too* deep.)

## Fibonacci: recursion's cautionary tale

The Fibonacci numbers ($0, 1, 1, 2, 3, 5, 8, \ldots$ — each the sum of the previous two) translate into gorgeous, doomed recursion. Let's count the calls:

In [ ]:
naive_calls = 0

def fib_naive(n):
    global naive_calls
    naive_calls += 1
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

for n in [10, 15, 20]:
    naive_calls = 0
    value = fib_naive(n)
    print(f"fib({n}) = {value:>5}   took {naive_calls:>6} calls")

Twenty-one thousand calls to compute `fib(20)`?! The trace would show `fib(10)` being recomputed over and over: the two branches overlap almost entirely, and the call count itself grows exponentially — add 5 to $n$ and the count multiplies by ~11.

The cure is **memoization** (yes, no "r"): cache each answer the first time you compute it, and return the cached copy ever after. One dict transforms the algorithm:

In [ ]:
memo_calls = 0

def fib_memo(n, cache=None):
    global memo_calls
    memo_calls += 1
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    cache[n] = n if n < 2 else fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    return cache[n]

for n in [10, 15, 20, 30]:
    memo_calls = 0
    value = fib_memo(n)
    print(f"fib({n}) = {value:>7}   took {memo_calls:>3} calls")

print("\nfib(20): naive 21891 calls  vs  memoized 39 calls")

From exponential to linear — $O(2^n)$-ish down to $O(n)$ — by *remembering*. Memoization (and its systematic cousin, dynamic programming) is one of the great algorithmic superpowers: many "impossible" problems are just naive recursion awaiting a cache.

## Sorting: selection vs insertion

Time to instrument two classic sorting algorithms with comparison counters.

**Selection sort** repeatedly finds the smallest remaining value and swaps it into place. It is beautifully predictable — and stubbornly blind: it compares every pair no matter what, even if the list is already sorted.

In [ ]:
def selection_sort(items):
    a = list(items)                     # sort a copy; count comparisons
    comparisons = 0
    for i in range(len(a)):
        best = i
        for j in range(i + 1, len(a)):
            comparisons += 1
            if a[j] < a[best]:
                best = j
        a[i], a[best] = a[best], a[i]
    return a, comparisons

**Insertion sort** works like sorting cards in your hand: take the next item and slide it left past everything bigger. Crucially, if nothing is bigger, it stops *immediately* — so its cost depends on how disordered the input is.

In [ ]:
def insertion_sort(items):
    a = list(items)
    comparisons = 0
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if a[j] > key:
                a[j + 1] = a[j]         # slide the bigger card right
                j -= 1
            else:
                break                   # found key's spot - stop early!
        a[j + 1] = key
    return a, comparisons

Now the shoot-out: both algorithms, three kinds of input — random, already sorted, and reversed. Predictions first! Which cells of the table will be big, which small?

In [ ]:
import random
random.seed(7)

n = 200
inputs = {
    "random":  random.sample(range(1000), n),
    "sorted":  list(range(n)),
    "reversed": list(range(n, 0, -1)),
}

print(f"comparisons at n={n}: | selection | insertion")
print("----------------------+-----------+----------")
for label, data in inputs.items():
    sel_result, sel_c = selection_sort(data)
    ins_result, ins_c = insertion_sort(data)
    assert sel_result == sorted(data) and ins_result == sorted(data)
    print(f"{label:>21} | {sel_c:>9} | {ins_c:>8}")

Selection sort: 19,900 comparisons ($= n(n-1)/2$) in every single row — it doesn't even notice the input is sorted. Insertion sort: identical worst case on reversed input, but just $n-1 = 199$ comparisons on sorted input — it *adapts*. That's why insertion sort is the real-world choice for small or nearly-sorted data (it runs inside Python's own sorting machinery for short runs!), while both are hopeless at scale: $O(n^2)$ is $O(n^2)$.

## Merge sort: divide and conquer

To beat quadratic sorting you must stop comparing (almost) everything with (almost) everything. **Merge sort**'s insight: half-sorting twice is cheaper than sorting once, *if* you can cheaply combine the halves. And you can — merging two already-sorted lists is one linear pass with two fingers:

In [ ]:
def merge(left, right):
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:           # take the smaller front item
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1
    merged.extend(left[i:])               # one side ran dry - take the rest
    merged.extend(right[j:])
    return merged

print(merge([2, 5, 9], [1, 5, 6, 11]))

With merging in hand, the sort is almost embarrassingly short: split in half, recursively sort each half, merge. The base case: a list of 0 or 1 items is already sorted.

In [ ]:
def merge_sort(items):
    if len(items) <= 1:
        return list(items)
    mid = len(items) // 2
    left = merge_sort(items[:mid])
    right = merge_sort(items[mid:])
    return merge(left, right)

import random
random.seed(7)
data = random.sample(range(1000), 12)
print("before:", data)
print("after: ", merge_sort(data))
assert merge_sort(data) == sorted(data)

The cost analysis, roughly: splitting in half over and over gives about $\log_2 n$ levels, and every level does a total of $n$ merge work — so $O(n \log n)$ overall. For a million items that's ~20 million steps versus selection sort's ~500 *billion*. Same task, different planet.

## Binary search — and the off-by-one minefield

Searching a *sorted* list shouldn't take $O(n)$: check the middle, discard the wrong half, repeat. Each probe halves the field — $O(\log n)$, the same magic as the BST.

Simple idea, notoriously bug-prone implementation. The published version in a 1986 textbook classic was wrong for two decades. The danger zone is the boundary bookkeeping:

In [ ]:
def binary_search(items, target):
    lo, hi = 0, len(items) - 1          # INCLUSIVE bounds: both ends in play
    while lo <= hi:                     # <= because a 1-item range is valid
        mid = (lo + hi) // 2
        if items[mid] == target:
            return mid
        elif items[mid] < target:
            lo = mid + 1                # +1: mid itself is already ruled out
        else:
            hi = mid - 1                # -1: same reason
    return -1                           # range emptied: not present

sorted_data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
for target in [23, 2, 91, 15]:
    print(f"binary_search for {target:>2} -> index {binary_search(sorted_data, target)}")

**The off-by-one discussion.** Three decisions in that code conspire, and each has a wrong-looking-right alternative:

- `hi = len(items) - 1` with `while lo <= hi` treats both bounds as *inclusive*. The equally valid alternative convention is `hi = len(items)` (exclusive) with `lo < hi` — but *mixing* the two conventions gives you a search that misses the last element or reads past the end.
- `lo = mid + 1` / `hi = mid - 1`: forget a `+1`/`-1` and a two-element range can stop shrinking — an **infinite loop**, the classic binary search death.
- The historical footnote: in fixed-width languages, `(lo + hi) // 2` can *overflow* when `lo + hi` exceeds the integer maximum — that was the actual bug that sat in production libraries for years. Python's unlimited integers make it a non-issue here, but the safe form `lo + (hi - lo) // 2` is worth recognising.

The defence is the same as always: test the *edges* — first element, last element, absent-but-in-range, absent-below, absent-above, empty list:

In [ ]:
data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]

assert binary_search(data, 2) == 0        # first
assert binary_search(data, 91) == 9       # last
assert binary_search(data, 15) == -1      # absent, inside the range
assert binary_search(data, -7) == -1      # absent, below everything
assert binary_search(data, 999) == -1     # absent, above everything
assert binary_search([], 5) == -1         # empty list
assert binary_search([42], 42) == 0       # single element, present
assert binary_search([42], 7) == -1       # single element, absent
print("All eight edge cases pass.")

## The complexity race

To close, the big picture in one chart: the standard growth curves side by side. The y-axis is logarithmic — it has to be, or $2^n$ would flatten every other curve against the floor within the first few pixels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = np.arange(1, 61)
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(n, np.ones_like(n), label="$O(1)$")
ax.plot(n, np.log2(n), label="$O(\\log n)$")
ax.plot(n, n, label="$O(n)$")
ax.plot(n, n * np.log2(np.maximum(n, 2)), label="$O(n \\log n)$")
ax.plot(n, n.astype(float) ** 2, label="$O(n^2)$")
ax.plot(n, 2.0 ** n, label="$O(2^n)$")

ax.set_yscale("log")
ax.set_xlabel("input size n")
ax.set_ylabel("steps (log scale)")
ax.set_title("The complexity race: every curve eventually loses to the one below it")
ax.legend(loc="upper left")
ax.grid(True, which="both", alpha=0.3)

On the log axis, $O(2^n)$ is the straight line racing off the top — by $n = 60$ it needs about $10^{18}$ steps, or several decades of computer time, while $O(n \log n)$ still sits under 400. **The gap between complexity classes dwarfs any hardware upgrade**: a computer 1000x faster moves an exponential algorithm's feasible $n$ up by just 10.

## What you just learned

- Count steps to find the growth shape; Big-O names it (drop constants and lesser terms).
- The doubling experiment: time at $n$ and $2n$ — the ratio betrays the exponent; log-log plots turn growth laws into visible slopes.
- Recursion = smaller self-calls + a base case; naive Fibonacci explodes; memoization collapses it to linear.
- Selection sort is always $n(n-1)/2$ comparisons; insertion sort adapts to nearly-sorted input; both are $O(n^2)$ — merge sort's divide-and-conquer achieves $O(n \log n)$.
- Binary search is $O(\log n)$ and an off-by-one minefield: pick one boundary convention and test the edges.

## Try it yourself

### Exercise 1 — The cubic counter

Predict first, then verify: write a step counter with **three** nested loops (`i`, then `j` from `i+1`, then `k` from `j+1` — all the *triples*). If $n = 20$ gives you some count, what should $n = 40$ give — roughly 2x, 4x, or 8x? Run both and check your prediction.

In [ ]:
def count_triple_checks(n):
    steps = 0
    # your code here: three nested loops counting every (i, j, k) triple
    return steps

print("n=20:", count_triple_checks(20))
print("n=40:", count_triple_checks(40))
# Uncomment when implemented:
# assert count_triple_checks(20) == 1140     # that's C(20,3)
# print("ratio:", count_triple_checks(40) / count_triple_checks(20))

### Exercise 2 — Memoize the grid walker

A robot walks from the top-left of an $r \times c$ grid to the bottom-right, moving only right or down. The path count obeys `paths(r, c) = paths(r-1, c) + paths(r, c-1)`, with 1 path for any grid that is 1 wide or 1 tall. Write it recursively **with a cache** (copy the `fib_memo` pattern). Without the cache, `paths(14, 14)` takes tens of millions of calls — with it, a few hundred.

In [ ]:
def paths(r, c, cache=None):
    # your code here (mind the base case: r == 1 or c == 1 -> 1)
    pass

# Uncomment to test:
# assert paths(1, 1) == 1
# assert paths(2, 2) == 2
# assert paths(3, 3) == 6
# print("paths(14, 14) =", paths(14, 14))    # should print 10400600, fast

### Exercise 3 — Leftmost match

Our `binary_search` returns *some* index of the target — if the value appears several times, no promises which. Write `binary_search_first(items, target)` returning the **leftmost** index: when you find a match, don't return — remember it and keep searching the *left* half (`hi = mid - 1`).

In [ ]:
def binary_search_first(items, target):
    lo, hi = 0, len(items) - 1
    found = -1
    # your code here: like binary_search, but on a match record mid and go left
    return found

# Uncomment to test:
# assert binary_search_first([1, 3, 3, 3, 7, 9], 3) == 1
# assert binary_search_first([3, 3, 3], 3) == 0
# assert binary_search_first([1, 2, 4], 3) == -1
# print("binary_search_first works!")

### Exercise 4 — List vs set: a doubling experiment

`x in my_list` scans ($O(n)$); `x in my_set` hashes ($O(1)$ on average). Prove it: for sizes 10,000 / 20,000 / 40,000, build both a list and a set of `range(size)`, then time many repeated lookups of a *missing* value (worst case for the list) using the provided timer. Which container's time doubles along with the size, and which stays flat?

In [ ]:
import time

def time_lookups(container, probe, repeats=200):
    start = time.perf_counter()
    for _ in range(repeats):
        probe in container            # the lookup under test
    return time.perf_counter() - start

# your code here: for size in [10_000, 20_000, 40_000], compare
#   time_lookups(list(range(size)), -1)  vs  time_lookups(set(range(size)), -1)

### Exercise 5 — Predict the trace

Without running: how many lines does `factorial(4)` print (using our tracing version from earlier), and what is the *first* line? Write your predictions as comments, then run and check.

In [ ]:
# prediction: number of lines = ?
# prediction: first line = ?
factorial(4)